# Chapter 3 · Transformers and LLMs — All Examples, Hands-on
**Social Media Analytics and Sentiment Analysis · SRH Heidelberg · Prof. Dr. Mehrdad Jalali**

This notebook runs **every example from the Chapter 3 slides with real models**. Each section has:
- 💡 **Idea**: what the slide explained, in two sentences
- ▶ **Run**: the code
- ✏️ **Try it**: a small test you change and run yourself

**Before you start:** *Runtime → Change runtime type → T4 GPU* (needed only for section 9). Then *Runtime → Run all*.
The first run downloads several models (≈ 3 GB) — this takes a few minutes.

| # | Section | Slide |
|---|---|---|
| 1 | Static vs. contextual embeddings | What Transformers Add |
| 2 | Self-attention | What Transformers Add |
| 3 | Pretraining: fill the gap | What Transformers Add |
| 4 | Subword tokenizer | What Transformers Add |
| 5 | Inside the model, step by step | Inside the Model |
| 6 | Users, links and the length limit | Tokens and Limits |
| 7 | Pretrained inference | Three Ways, One Post |
| 8 | Zero-shot: the label words matter | Zero-Shot Depends on the Label Words |
| 9 | Fine-tuning: before and after | Fine-Tuning — Before and After |
| 10 | Weak vs. good prompt | A Weak Prompt vs. a Good Prompt |
| 11 | Checking a model against humans (Cohen's κ) | Checking an LLM Against Humans |

> Numbers on the slides were illustrative. **Your numbers here are real** — they may differ, and that is the point: always check.

In [ ]:
!pip -q install -U transformers accelerate sentencepiece
import torch, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          pipeline, logging)
logging.set_verbosity_error()
DEVICE = 0 if torch.cuda.is_available() else -1
print("GPU available:", torch.cuda.is_available())
pd.set_option("display.max_colwidth", 80)

## 1 · Static vs. contextual embeddings
💡 **Idea.** A *static* embedding gives a word **one vector**, whatever the sentence. A *contextual* embedding (BERT) computes the vector **from the whole sentence** — so “sick” can mean *ill* or *great*.

We use BERT twice: its **input embedding table** (static — the same for every sentence) and its **last layer** (contextual).

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True).eval()
cos = torch.nn.functional.cosine_similarity

def word_vectors(sentence, word):
    enc = tok(sentence, return_tensors="pt")
    ids = enc["input_ids"][0].tolist()
    pos = ids.index(tok.convert_tokens_to_ids(word))          # position of the word
    with torch.no_grad():
        out = bert(**enc)
    static = bert.embeddings.word_embeddings.weight[ids[pos]]  # same for every sentence
    contextual = out.last_hidden_state[0, pos]                 # depends on the sentence
    return static, contextual

s1 = "i feel sick and want to stay in bed"
s2 = "this new camera is sick , the photos look amazing"
st1, cx1 = word_vectors(s1, "sick")
st2, cx2 = word_vectors(s2, "sick")
print("similarity of 'sick' in the two sentences")
print("  static     :", round(cos(st1, st2, dim=0).item(), 3))
print("  contextual :", round(cos(cx1, cx2, dim=0).item(), 3))

In [ ]:
# Which meaning is closer? Compare with 'ill' and 'great' in neutral sentences
_, ill = word_vectors("he was ill for a week", "ill")
_, great = word_vectors("the food was great", "great")
for name, v in [("'sick' in sentence 1", cx1), ("'sick' in sentence 2", cx2)]:
    print(f"{name}:  vs ill = {cos(v, ill, dim=0):.3f}   vs great = {cos(v, great, dim=0):.3f}")

✏️ **Try it.** Replace `sick` with another word that changes meaning, e.g. **“fire”** (“the house is on fire” vs. “this song is fire”) or **“bad”**. Does the contextual similarity drop below the static one (always 1.0)?

## 2 · Self-attention
💡 **Idea.** In every layer, each word **looks at every other word** and decides how much it matters. In “not really happy”, the word *happy* should look at *not*.

We print the attention from one word to all others (last layer, averaged over the 12 heads).

In [ ]:
def attention_from(sentence, word, layer=-1):
    enc = tok(sentence, return_tensors="pt")
    tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
    with torch.no_grad():
        att = bert(**enc).attentions[layer][0].mean(0)   # heads averaged -> [tokens x tokens]
    i = tokens.index(word)
    row = att[i].numpy()
    return pd.DataFrame({"token": tokens, "attention": row.round(3)}).iloc[1:-1]  # drop [CLS]/[SEP]

df = attention_from("i am not really happy with the charger", "happy")
df.style.bar(subset=["attention"], color="#c9b6f0")

✏️ **Try it.** Change `layer=-1` to `layer=2` or `layer=6`. Does *happy* look at *not* more in early or in late layers? Try the sentence “the battery is not bad at all” and look from **“bad”**.

*Note: attention is a hint, not a full explanation of the model's decision.*

## 3 · Pretraining: fill the gap
💡 **Idea.** Models like RoBERTa learned language by **predicting hidden words** in billions of sentences — no labels needed. We can ask the pretrained model to fill a gap.

In [ ]:
fill = pipeline("fill-mask", model="roberta-base", device=DEVICE)
for r in fill("Great, the train is late <mask> 🙄", top_k=5):
    print(f"{r['token_str'].strip():12s} {r['score']:.2f}")

✏️ **Try it.** Write your own social-media sentence with one `<mask>`, e.g. `"Charging took forever, this app is <mask>"` or `"Best road trip ever <mask>"`. Does the model guess an emoji, a word or a hashtag?

## 4 · Subword tokenizer
💡 **Idea.** Rare or new words are split into **known pieces**, so no word is ever “unknown”. BERT marks continuation pieces with `##`; RoBERTa marks a **leading space** with `Ġ`.

In [ ]:
rob_tok = AutoTokenizer.from_pretrained("roberta-base")
for w in ["sooo slow", "#BatteryGate", "rangeanxiety", "Ladesäule", "😡"]:
    print(f"{w:15s} BERT: {tok.tokenize(w)}")
    print(f"{'':15s} RoBERTa: {rob_tok.tokenize(w)}")

✏️ **Try it.** Tokenize slang (`"ngl this is mid"`), a long hashtag (`"#WinterRangeTest2026"`) or a German compound. Which tokenizer splits into more pieces?

## 5 · Inside the model, step by step
💡 **Idea.** Post → tokens → token IDs → transformer layers → three scores. We use a sentiment model trained on ~124 million tweets.

In [ ]:
SENT = "cardiffnlp/twitter-roberta-base-sentiment-latest"
s_tok = AutoTokenizer.from_pretrained(SENT)
s_model = AutoModelForSequenceClassification.from_pretrained(SENT).eval()

post = "Not really happy with the charger 😡"
enc = s_tok(post, return_tensors="pt")
print("1 · post     :", post)
print("2 · tokens   :", s_tok.convert_ids_to_tokens(enc["input_ids"][0]))
print("3 · token IDs:", enc["input_ids"][0].tolist())
with torch.no_grad():
    logits = s_model(**enc).logits[0]
print("4 · layers   :", s_model.config.num_hidden_layers, "transformer layers")
probs = torch.softmax(logits, -1)
print("5 · scores   :", {s_model.config.id2label[i]: round(p.item(), 3) for i, p in enumerate(probs)})

✏️ **Try it.** Remove the emoji, or change “Not really happy” to “Really happy”. How much does each change move the scores?

## 6 · Users, links and the length limit
💡 **Idea.** Twitter models were trained with names replaced by `@user` and links by `http`. Do the same — it also protects privacy. And BERT-type models read **at most 512 tokens**.

In [ ]:
def preprocess(text):                       # the convention used by the Cardiff models
    return " ".join("@user" if t.startswith("@") and len(t) > 1 else
                    "http" if t.startswith("http") else t for t in text.split())

clf = pipeline("sentiment-analysis", model=SENT, tokenizer=s_tok, device=DEVICE)
raw = "@CityRail your app crashed again https://t.co/k2x"
print(preprocess(raw))
print(clf(preprocess(raw)))

long_post = "The car is fine. " * 150 + "But overall I hate the charging network."
n = len(s_tok(long_post)["input_ids"])
print("\ntokens in the long post:", n, "| model limit:", s_tok.model_max_length)
print("truncated (end lost):", clf(long_post, truncation=True, max_length=512))
print("only the end       :", clf(long_post[-200:]))

✏️ **Try it.** Move the important sentence to the **beginning** of `long_post`. Does the truncated prediction change? What does that tell you about long Reddit comments?

## 7 · Pretrained inference
💡 **Idea.** Use a model that is **already fine-tuned** — one line, works on CPU. But you get **its labels**, not yours.

In [ ]:
posts = ["Charging takes forever, but I still love driving electric.",
         "Waiting 40 min at the charger. Love it.",
         "Not the worst update, honestly.",
         "New firmware 2.4 out today #EV",
         "Das Update ist echt mies"]
res = clf([preprocess(p) for p in posts])
pd.DataFrame({"post": posts, "label": [r["label"] for r in res], "score": [round(r["score"], 2) for r in res]})

✏️ **Try it.** Add three posts from a public Reddit or YouTube thread (no user names). Which ones would you label differently?

## 8 · Zero-shot: the label words matter
💡 **Idea.** An NLI model checks whether the post supports the sentence *“This example is {label}.”* — **no training**. But the **wording of the labels** changes the answer.

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEVICE)
post = "EVs are fine, but charging still drives me crazy."
wordings = [["pro EV", "anti EV", "neutral"],
            ["supports electric cars", "opposes electric cars", "neutral"],
            ["in favour of EVs", "against EVs", "no position"]]
rows = []
for labels in wordings:
    r = zs(post, candidate_labels=labels)
    rows.append({"labels": " · ".join(labels), "top label": r["labels"][0], "score": round(r["scores"][0], 2)})
pd.DataFrame(rows)

In [ ]:
# The hypothesis template is part of the wording too
for tpl in ["This example is {}.", "The author of this post is {}."]:
    r = zs(post, candidate_labels=wordings[2], hypothesis_template=tpl)
    print(f"{tpl:35s} -> {r['labels'][0]} ({r['scores'][0]:.2f})")

✏️ **Try it.** Write your own label set for a new question, e.g. *Is the post about charging, price or range?* Try two wordings. Do they agree? This is why zero-shot **always** needs validation on labelled posts.

## 9 · Fine-tuning: before and after
💡 **Idea.** Adapt a pretrained model to **your labelled data**. It learns your domain — but only what is in your labels.

We fine-tune DistilBERT on the English posts of the course dataset `ev_posts.csv` and compare it with the general Twitter model on five hand-written posts.
⚠️ Needs a **GPU** (≈ 2–4 min). The course data is template-generated, so training scores look very high — judge the model on the **new** posts at the end.

In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/SocialMediaAnayltics/main/data/ev_posts.csv"
df = pd.read_csv(URL)
df = df[df["lang"] == "en"].drop_duplicates("text")
labels = ["negative", "neutral", "positive"]; lab2id = {l: i for i, l in enumerate(labels)}
tr, va = train_test_split(df, test_size=0.2, random_state=42, stratify=df["sentiment"])

FT = "distilbert-base-uncased"
ft_tok = AutoTokenizer.from_pretrained(FT)
class Posts(Dataset):
    def __init__(self, d):
        self.enc = ft_tok(list(d["text"]), truncation=True, max_length=64, padding="max_length")
        self.y = [lab2id[s] for s in d["sentiment"]]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i]); return item

ft_model = AutoModelForSequenceClassification.from_pretrained(FT, num_labels=3,
            id2label=dict(enumerate(labels)), label2id=lab2id)
def metrics(p):
    return {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}
args = TrainingArguments("ft_out", num_train_epochs=2, per_device_train_batch_size=32,
                         eval_strategy="epoch", logging_strategy="epoch", save_strategy="no",
                         learning_rate=5e-5, seed=42, report_to=[])
trainer = Trainer(model=ft_model, args=args, train_dataset=Posts(tr), eval_dataset=Posts(va), compute_metrics=metrics)
trainer.train()

In [ ]:
new_posts = ["Range anxiety is real this winter",
             "Recall fixed in two days, impressive",
             "Charger finally works again",
             "Dealer says update on Monday",
             "Waiting 40 min at the charger. Love it."]
true = ["negative", "positive", "positive", "neutral", "negative"]
ft_pipe = pipeline("text-classification", model=trainer.model, tokenizer=ft_tok, device=DEVICE)
general = [r["label"] for r in clf(new_posts)]
tuned = [r["label"] for r in ft_pipe(new_posts)]
mark = lambda p, t: ("✓ " if p == t else "✗ ") + p
pd.DataFrame({"post": new_posts, "true": true,
              "general model": [mark(p, t) for p, t in zip(general, true)],
              "fine-tuned": [mark(p, t) for p, t in zip(tuned, true)]})

✏️ **Try it.** Add five posts of your own to `new_posts`. Where does the fine-tuned model fail — and does the training data contain anything like those posts? Change `num_train_epochs` to 6: does validation macro-F1 still improve (overfitting)?

## 10 · Weak prompt vs. good prompt
💡 **Idea.** How you ask decides what you get. A good prompt has **fixed labels, a definition, one example and a fixed output format**.
We use the small open model **FLAN-T5** (no API key needed). Large LLMs behave the same way — only better.

In [ ]:
from transformers import AutoModelForSeq2SeqLM
t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").eval()
def ask(prompt):
    enc = t5_tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out = t5.generate(**enc, max_new_tokens=30, do_sample=False)   # temperature 0
    return t5_tok.decode(out[0], skip_special_tokens=True)

post = "Waiting 40 min at the charger. Love it."
weak = f"What is the sentiment of this post? {post}"
good = ("Classify the sentiment of a social media post. "
        "Labels: positive, neutral, negative. "
        "Sarcasm counts as the opposite of the literal words. "
        "Example: 'Great, train late again' -> negative. "
        f"Answer with one label only.\nPost: {post}\nLabel:")
for name, p in [("weak", weak), ("good", good)]:
    print(f"{name:5s} ->", ask(p))

✏️ **Try it.** Remove the sarcasm sentence from the good prompt — does the answer change? Run the weak prompt on all five posts from section 7 and count how many answers you could put into a table without editing.

*Rules for LLM labelling:* fix the model version, use `do_sample=False` (temperature 0), and save every prompt for your disclosure.

## 11 · Checking a model against humans (Cohen's κ)
💡 **Idea.** Accuracy counts agreement — but some agreement happens **by luck**. Cohen's κ removes it.
κ = (pₒ − pₑ) / (1 − pₑ). Rough guide: 0.41–0.60 moderate, 0.61–0.80 substantial.

In [ ]:
from collections import Counter
from sklearn.metrics import cohen_kappa_score
human = ["neg","neg","pos","neu","neg","pos","neu","neg","pos","neu"]
model = ["neg","neg","pos","pos","neg","pos","neg","neg","pos","neu"]

po = np.mean([h == m for h, m in zip(human, model)])
ch, cm = Counter(human), Counter(model)
pe = sum(ch[k] * cm[k] for k in ch) / len(human) ** 2
print(f"observed agreement p_o = {po:.2f}")
print(f"chance agreement   p_e = {pe:.2f}")
print(f"kappa (by hand)        = {(po - pe) / (1 - pe):.3f}")
print(f"kappa (scikit-learn)   = {cohen_kappa_score(human, model):.3f}")
print("disagreements at posts:", [i + 1 for i, (h, m) in enumerate(zip(human, model)) if h != m])

In [ ]:
# ✏️ Your turn: label the five posts from section 7 yourself, then compare with the model
my_labels = ["negative", "negative", "positive", "neutral", "negative"]   # <- change to your own labels
model_labels = [r["label"] for r in res]
print("agreement:", np.mean([a == b for a, b in zip(my_labels, model_labels)]))
print("kappa    :", round(cohen_kappa_score(my_labels, model_labels), 3))

✏️ **Try it.** Ask a neighbour to label the same five posts **without looking** at yours. Compute κ between the two of you — humans disagree too!

---
### Summary
| Idea | What you saw |
|---|---|
| Contextual embeddings | the same word gets different vectors in different sentences |
| Self-attention | words look at the words that change their meaning |
| Pretraining | the model fills gaps it has never been taught explicitly |
| Subwords | new words are split into known pieces |
| Inference / zero-shot / fine-tuning | three ways to use a model — each must be **validated** |
| Prompts and κ | ask precisely, then check against humans |

**Context is what lexicons and TF-IDF were missing — but every model still needs to be checked on labelled posts.**